In [ ]:
# Imports

import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# File declarations

FILES_BASE_DIR = "data/"

FILES_FLOW_DIR = FILES_BASE_DIR + "flow/"
FILES_FLOW = [
    "Krapina-Kupljenovo_2000-2024.csv",
    "Krapina-Kupljenovo_2025.csv",
    # "Krapina-ZlatarBistrica_2000-2024.csv",
    # "Krapinica-Zabok_2000-2024.csv",
    # "Krapinica-Zabok_2025.csv",
]

FILES_PRECIPITATION_DIR = FILES_BASE_DIR + "precipitation/"
FILES_PRECIPITATION = [
    # "Krapina_2000-2020.csv",
    # "Krapina_2021-2025_unconfirmed.csv",
    "Puntijarka_2000_unconfirmed.csv",
    "Puntijarka_2000-2020.csv",
    "Puntijarka_2020-2025_unconfirmed.csv",
    # "Varazdin_2000-2024.csv",
    # "Varazdin_2024-2025_unconfirmed.csv",
    # "Zagreb-Maksimir_2000-2025.csv",
]

FILES_TEMPERATURE_DIR = FILES_BASE_DIR + "temperature/"
FILES_TEMPERATURE = [
    # "Krapina_2000-2020.csv",
    # "Krapina_2021-2025_unconfirmed.csv",
    "Puntijarka_2000_unconfirmed.csv",
    "Puntijarka_2000-2020.csv",
    "Puntijarka_2020-2025_unconfirmed.csv",
    # "Varazdin_2000-2024.csv",
    # "Varazdin_2024-2025_unconfirmed.csv",
    # "Zagreb-Maksimir_2000-2025.csv",
]

FILE_SM2_WEIGHTS = "weights_SM2.csv"

In [ ]:
# Check if all declared files exist

checks = []

for base_dir, files in [
    (FILES_FLOW_DIR, FILES_FLOW),
    (FILES_PRECIPITATION_DIR, FILES_PRECIPITATION),
    (FILES_TEMPERATURE_DIR, FILES_TEMPERATURE),
]:
    for filename in files:
        path = Path(base_dir) / filename
        if not path.exists():
            checks.append(str(path))

if not checks:
    print("All declared files exist.")
else:
    missing_df = pd.DataFrame(checks, columns=["Missing declared files"])
    print(missing_df)
    raise FileNotFoundError("Some declared files are missing.")

In [ ]:
def parse_flow_data(file_path):
    """
     Returns:
        {
            "metadata": dict,
            "data": pandas.DataFrame
        }
    """

    file_path = Path(file_path)
    with open(file_path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]

    metadata = {}
    data_start_idx = None
    for i, line in enumerate(lines):
        # Detect start of data
        if ":" not in line:
            data_start_idx = i
            break

        # Metadata lines
        key, value = line.split(":", 1)
        metadata[key.strip()] = value.strip()

    if data_start_idx is None:
        raise RuntimeError("Data section not found in file: " + str(file_path))

    # Data section
    df = pd.read_csv(
        file_path,
        sep=";",
        skiprows=data_start_idx,
        decimal=","
    )

    df.columns = [c.strip() for c in df.columns]

    df["datetime"] = pd.to_datetime(
        df["Date"] + " " + df["Time"],
        format="%d.%m.%Y. %H:%M:%S"
    )

    value_col = [c for c in df.columns if "Value" in c][0]
    df = df.rename(columns={value_col: "value"})

    df = df[["datetime", "value"]]

    return {
        "metadata": metadata,
        "data": df
    }


def parse_meteo_data(file_path):
    """
    Returns:
    {
        "metadata": dict,
        "data": pandas.DataFrame
    }
    """

    file_path = Path(file_path)

    with open(file_path, "r", encoding="utf-8") as f:
        lines = [next(f).strip() for _ in range(3)]

    metadata = {}

    has_metadata = "DDMMYYYY" in lines[1]

    data_start_idx = 0

    if has_metadata:
        first = lines[0].split(";")
        second = lines[1].split(";")

        metadata = {
            "station_id": first[0],
            "station_name": first[1],
            "latitude": first[2],
            "longitude": first[3],
            "Hs": first[4],
            "Hb": first[5],
            "parameter": second[2].strip(),
        }

        # data starts after:
        # metadata line
        # header line
        # units line
        data_start_idx = 3

    # Data section
    df = pd.read_csv(
        file_path,
        sep=";",
        skiprows=data_start_idx,
        header=None,
        usecols=[0, 1, 2],
        names=["hour", "date", "value"],
        dtype=str
    )

    # Prepend zeros to ensure correct parsing and properly transform 1-24h to 0-23h
    df["date"] = df["date"].str.zfill(8)
    hour = df["hour"].astype(int)
    base_datetime = pd.to_datetime(
        df["date"],
        format="%d%m%Y",
        errors="coerce"
    )
    df["datetime"] = base_datetime + pd.to_timedelta(hour, unit="h")

    df["value"] = pd.to_numeric(
        df["value"].str.replace(",", ".", regex=False),
        errors="coerce",
    )

    df = df[["datetime", "value"]]

    return {
        "metadata": metadata,
        "data": df
    }

In [ ]:
def reformat_precipitation_data(df):
    # Empty means 0 precipitation
    # -999 means missing data
    df["value"] = df["value"].replace({np.nan: 0, -999: np.nan})
    return df

def load_all_and_analyze(base_dir, files, parser, reformatter=None):
    print(f"### LOADING FILES FROM: {base_dir} ###")
    datas = []
    for f in files:
        result = parser(Path(base_dir) / f)
        if reformatter:
            result["data"] = reformatter(result["data"])
        print(f"Parsed file: {f}")
        datas.append(result)

        print("Metadata:")
        for k, v in result["metadata"].items():
            print(f"\t{k}: {v}")
        print("\n")

    data = pd.concat([d["data"] for d in datas], ignore_index=True)

    # Basic analysis
    print("# Combined data analysis:")
    count_total = len(data)
    count_invalid = data["value"].isna().sum()
    count_duplicates = data.duplicated(subset=["datetime"]).sum()
    desc = data["value"].describe()
    start_date = data["datetime"].min()
    end_date = data["datetime"].max()
    kurtosis = data["value"].kurtosis()
    q1 = data["value"].quantile(0.25)
    q3 = data["value"].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = data[(data["value"] < lower_bound) | (data["value"] > upper_bound)]
    outlier_count = len(outliers)
    delta_times = data["datetime"].diff().dt.total_seconds()
    most_common_delta = delta_times.mode()[0]
    non_common_delta_count = (delta_times != most_common_delta).sum()
    largest_deltas = delta_times.sort_values(ascending=False).head(5)
    largest_deltas_spans = [
        (data.loc[i - 1, "datetime"], data.loc[i, "datetime"])
        for i in largest_deltas.index
        if i > 0
    ]

    print(f"Start date: {start_date}")
    print(f"End date: {end_date}")
    print(f"Total records: {count_total}")
    print(f"Invalid records: {count_invalid} ({(count_invalid / count_total):.2%})")
    print(f"Duplicate records: {count_duplicates} ({(count_duplicates / count_total):.2%})")
    print(f"Value description:\n{desc}")
    print(f"Kurtosis: {kurtosis:.4f}")
    print(f"Outlier count: {outlier_count}")
    print(f"Outlier lower bound: {lower_bound}")
    print(f"Outlier upper bound: {upper_bound}")
    print(f"Delta times (seconds): {delta_times.describe()}")
    print(f"Most common delta: {most_common_delta}")
    print(f"Non-common delta count: {non_common_delta_count}")
    print(f"Largest delta spans:")
    for start, end in largest_deltas_spans:
        print(f"\t{start} -> {end}")
    print("\n\n")

    return data

In [ ]:
load_all_and_analyze(FILES_FLOW_DIR, FILES_FLOW, parse_flow_data)
load_all_and_analyze(FILES_PRECIPITATION_DIR, FILES_PRECIPITATION, parse_meteo_data, reformat_precipitation_data)
load_all_and_analyze(FILES_TEMPERATURE_DIR, FILES_TEMPERATURE, parse_meteo_data)
False